
# Example 1 — Gaussian terminal condition in $d=2$: house $\to$ flower

This notebook instantiates the **Gaussian terminal condition** from Section 3.3 / **Algorithm 1** of the manuscript and turns the paper's short validation example into a visual $d=2$ toy problem.

Here:
- the initial labeled atomic configuration is a stylized **house**
- the terminal target configuration is a stylized **flower**
- particle **mass is frozen** and shown by color
- the main output is an **interactive time slider** (with play/pause controls)

The simulation uses the wrapped torus drift so the particles stay on $[0,1)^2$.


In [ ]:
import numpy as np
import sys
from pathlib import Path
from scipy.optimize import linear_sum_assignment


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.wasserstein_conditioning_algorithms import (
    shortest_periodic_displacement,
    simulate_gaussian_terminal_em,
)

np.set_printoptions(precision=3, suppress=True)


In [ ]:
import plotly.graph_objects as go

from notebooks.support import (
    center_trace,
    circle_trace,
    configure_plotly,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)

configure_plotly()


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=6):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=18,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        mass_format=".3f",
        time_formatter=lambda t, h: f"time = {t:.3f}",
        slider_label_formatter=lambda t, h: f"{t:.3f}",
        currentvalue_prefix="time = ",
        width=800,
        height=700,
        play_frame_duration=110,
    )


In [ ]:

# --- Stylized house and flower point sets on the flat torus [0, 1)^2 ---
# House points:
house_points = np.array([
    [0.28, 0.20],  # bottom left
    [0.50, 0.20],  # bottom middle
    [0.72, 0.20],  # bottom right
    [0.28, 0.38],  # left wall
    [0.72, 0.38],  # right wall
    [0.28, 0.56],  # top left
    [0.72, 0.56],  # top right
    [0.50, 0.78],  # roof apex
    [0.50, 0.56],  # roof base
    [0.42, 0.32],  # door left
    [0.58, 0.32],  # door right
    [0.50, 0.44],  # door top
], dtype=float)

# Flower points:
flower_points = np.array([
    [0.50, 0.62],  # flower center
    [0.50, 0.78],  # top petal
    [0.62, 0.72],  # upper-right petal
    [0.68, 0.60],  # right petal
    [0.62, 0.48],  # lower-right petal
    [0.50, 0.42],  # lower petal
    [0.38, 0.48],  # lower-left petal
    [0.32, 0.60],  # left petal
    [0.38, 0.72],  # upper-left petal
    [0.50, 0.28],  # stem top
    [0.50, 0.16],  # stem bottom
    [0.38, 0.22],  # leaf
], dtype=float)

# Reference outlines used only for plotting:
house_outline = house_points[[0, 1, 2, 4, 6, 7, 5, 3]]
house_door = house_points[[9, 11, 10]]

flower_ring = flower_points[[1, 2, 3, 4, 5, 6, 7, 8]]
flower_center = flower_points[[0]]
flower_stem = flower_points[[0, 9, 10]]
flower_leaf = flower_points[[9, 11]]

# Smoothly varying masses so the colorbar is informative, but not so uneven that
# the tiniest particles become numerically wild.
raw_masses = np.array([1.8, 1.6, 1.4, 1.0, 1.0, 0.9, 0.9, 1.7, 1.1, 0.8, 0.8, 1.0], dtype=float)
masses = raw_masses / raw_masses.sum()

# Because the Gaussian target is label-aware, we assign house points to flower points
# by a minimal Euclidean pairing before simulating.
cost_matrix = np.sum((house_points[:, None, :] - flower_points[None, :, :]) ** 2, axis=-1)
row_ind, col_ind = linear_sum_assignment(cost_matrix)

target_positions = np.empty_like(flower_points)
target_positions[row_ind] = flower_points[col_ind]

# Simulation parameters (chosen to give a visually readable house -> flower transition).
lambda_ = 200000
horizon = 0.0005
step_size = horizon/300
seed =33

print("number of particles:", len(masses))
print("mass range:", (float(masses.min()), float(masses.max())))
print("horizon:", horizon, "step_size:", step_size, "seed:", seed)


In [ ]:

rng = np.random.default_rng(seed)

sim = simulate_gaussian_terminal_em(
    masses=masses,
    target_positions=target_positions,
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    initial_positions=house_points,
    drift_mode="wrapped",
    image_radius=1,
    rng=rng,
    store_drifts=True,
)

print("positions array shape:", sim.positions.shape)
print("final time:", float(sim.times[-1]))


In [ ]:

static_traces = [
    line_trace(house_outline, name="house outline", color="rgba(30, 144, 255, 0.55)", dash="dash", close=True),
    line_trace(house_door, name="house door", color="rgba(30, 144, 255, 0.45)", dash="dash", close=False, showlegend=False),
    line_trace(flower_ring, name="flower outline", color="rgba(220, 20, 60, 0.55)", dash="dot", close=True),
    center_trace(flower_center, name="flower center", color="rgba(220, 20, 60, 0.75)", symbol="x", size=12, showlegend=False),
    line_trace(flower_stem, name="flower stem", color="rgba(34, 139, 34, 0.55)", dash="dot", close=False, showlegend=False),
    line_trace(flower_leaf, name="flower leaf", color="rgba(34, 139, 34, 0.55)", dash="dot", close=False, showlegend=False),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title="Example 1: Gaussian terminal condition (house → flower)",
    static_traces=static_traces,
    marker_size=18,
)
fig.show()


In [ ]:

# Weighted mean periodic distance to the assigned flower targets.
periodic_distances = np.sqrt(
    np.sum(
        shortest_periodic_displacement(sim.positions, target_positions[None, :, :]) ** 2,
        axis=-1,
    )
)
weighted_mean_distance = periodic_distances @ sim.masses

import plotly.graph_objects as go

distance_fig = go.Figure()
distance_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=weighted_mean_distance,
        mode="lines",
        name="weighted mean periodic distance",
    )
)
distance_fig.update_layout(
    title="How close the particles are to their assigned flower targets",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted mean periodic distance",
)
distance_fig.show()

print("final weighted mean periodic distance:", float(weighted_mean_distance[-1]))
